# ARIA — DevOps Incident Response: GRPO Training (Kaggle 2×T4)

**Model:** `unsloth/Llama-3.2-3B-Instruct` (4-bit quantized)  
**Tasks:** `easy` → `medium`  
**Episodes:** 80 per task (160 total)  
**Expected runtime:** ~6–8 hours on Kaggle 2×T4  

### Before running:
1. Enable **GPU T4 x2** (right panel → Accelerator)
2. Add Kaggle secret: Settings → Secrets → `HF_TOKEN` = your HF write token
3. Run all cells top to bottom — do not skip any

In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────────────────
import subprocess, sys, os

os.environ['UNSLOTH_RETURN_LOGITS'] = '1'

# Install in correct order
subprocess.run(['pip', 'install', '-q',
    'unsloth',
    'transformers>=4.48.0',   # needs 4.48+ for CompileConfig
    'mergekit',
    'trl>=0.9.0',
    'accelerate>=0.26.0',
    'peft>=0.10.0',
    'bitsandbytes',
    'requests',
    'matplotlib',
    'huggingface_hub',
], capture_output=True)

# Clear stale cache
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['trl','unsloth','transformers','peft']):
        del sys.modules[mod]

# Verify
import unsloth
from unsloth import FastLanguageModel
import transformers, peft, torch
from trl import GRPOConfig

print(f'✅ unsloth {unsloth.__version__}')
print(f'✅ transformers {transformers.__version__}')
print(f'✅ torch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
print(f'✅ UNSLOTH_RETURN_LOGITS = {os.environ["UNSLOTH_RETURN_LOGITS"]}')
print('✅ All good — proceed')

In [ ]:
# ── Cell 2: Authenticate HuggingFace ─────────────────────────────────────────
import os
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print('✅ HF token loaded from Kaggle secrets')
except Exception as e:
    # Fallback: paste token directly (remove before sharing notebook)
    hf_token = 'YOUR_HF_WRITE_TOKEN_HERE'
    os.environ['HF_TOKEN'] = hf_token
    print(f'⚠️  Using hardcoded token (Kaggle secret not found: {e})')

from huggingface_hub import login
login(token=hf_token, add_to_git_credential=False)
print('✅ Logged in to HuggingFace Hub')

In [ ]:
# ── Cell 3: Config ────────────────────────────────────────────────────────────
CONFIG = {
    'model_name': 'unsloth/Llama-3.2-3B-Instruct',
    'max_seq_length': 2048,
    'load_in_4bit': True,
    'env_url': 'https://arijit-07-devops-incident-response.hf.space',
    'tasks': ['easy', 'medium'],
    'episodes_per_task': 80,
    'max_steps_per_episode': 12,
    'learning_rate': 5e-6,
    'grpo_group_size': 6,
    'lora_rank': 16,
    'lora_alpha': 32,
    'kl_coeff': 0.05,
    'hf_repo': 'Arijit-07/aria-devops-llama3b',
    'output_dir': '/kaggle/working/aria-llama3b',
    'save_every_n_episodes': 20,
}
print('✅ Config loaded')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Cell 4: Environment Client ────────────────────────────────────────────────
import requests
import json
import time

BASE_URL = CONFIG['env_url']

def env_reset(task_id: str, seed: int = None) -> dict:
    payload = {'task_id': task_id}
    if seed is not None:
        payload['seed'] = seed
    for attempt in range(3):
        try:
            r = requests.post(f'{BASE_URL}/reset', json=payload, timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt == 2:
                raise
            time.sleep(5)

def env_step(action: dict) -> dict:
    for attempt in range(3):
        try:
            r = requests.post(f'{BASE_URL}/step', json=action, timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt == 2:
                raise
            time.sleep(5)

def env_state() -> dict:
    r = requests.get(f'{BASE_URL}/state', timeout=30)
    r.raise_for_status()
    return r.json()

# Test connection
health = requests.get(f'{BASE_URL}/health', timeout=15).json()
print(f'✅ Environment connected: {health}')

# Test reset
test_obs = env_reset('easy', seed=0)
print(f'✅ Reset successful. Task: {test_obs.get("task_id")}')
print(f'   Services: {len(test_obs.get("services", []))}')
print(f'   Alerts: {len(test_obs.get("active_alerts", []))}')

In [ ]:
# ── Cell 5: Observation → Text (what the LLM sees) ───────────────────────────
def observation_to_prompt(obs: dict, task_id: str) -> str:
    """Convert environment observation to LLM prompt text."""
    lines = []
    lines.append('=== PRODUCTION INCIDENT RESPONSE ===')
    lines.append(f'Task: {task_id.upper()} | Step: {obs.get("step", 0)}/{obs.get("max_steps", 15)}')
    lines.append('')

    # SLA Status
    sla = obs.get('sla_status', {})
    if sla:
        lines.append('SLA STATUS:')
        for svc, status in sla.items():
            emoji = '🔴' if status == 'breached' else '🟡' if status == 'warning' else '🟢'
            lines.append(f'  {emoji} {svc}: {status}')
        lines.append('')

    # Active Alerts
    alerts = obs.get('active_alerts', [])
    if alerts:
        lines.append('ACTIVE ALERTS:')
        for a in sorted(alerts, key=lambda x: x.get('severity',''), reverse=True):
            lines.append(f'  [{a.get("severity","").upper()}] {a.get("service","")}: {a.get("message","")}')
        lines.append('')

    # Services
    services = obs.get('services', [])
    if services:
        lines.append('SERVICE METRICS:')
        for s in sorted(services, key=lambda x: x.get('error_rate', 0), reverse=True):
            lines.append(
                f'  {s.get("name",""):30s} | status={s.get("status",""):10s} | '
                f'cpu={s.get("cpu",0):5.1f}% | mem={s.get("memory",0):5.1f}% | '
                f'err={s.get("error_rate",0):.3f} | p99={s.get("latency_p99",0):.0f}ms'
            )
        lines.append('')

    # Recent logs (partial — only 2 lines shown)
    logs = obs.get('recent_logs', {})
    if logs:
        lines.append('RECENT LOGS (partial — use read_logs for full history):')
        for svc, log_lines in list(logs.items())[:4]:
            for log_line in log_lines[:2]:
                lines.append(f'  [{svc}] {log_line}')
        lines.append('')

    # Service dependencies
    deps = obs.get('service_dependencies', [])
    if deps:
        lines.append('SERVICE DEPENDENCIES:')
        for d in deps[:6]:
            lines.append(f'  {d.get("service","")} → calls → {d.get("depends_on","")}')
        lines.append('')

    # Evidence log
    evidence = obs.get('evidence_log', [])
    if evidence:
        lines.append('EVIDENCE GATHERED THIS EPISODE:')
        for e in evidence[-5:]:  # last 5 evidence entries
            lines.append(f'  [{e.get("action_type","").upper()}] {e.get("content","")[:150]}')
        lines.append('')

    # Last result
    if obs.get('last_action_result'):
        lines.append(f'LAST ACTION RESULT: {obs["last_action_result"][:200]}')
        lines.append('')

    return '\n'.join(lines)


SYSTEM_PROMPT = """You are an expert DevOps engineer responding to a production incident.
Analyze the situation carefully and take the most appropriate action.

Available actions (respond with EXACTLY one JSON object):
- Read logs: {"action_type": "read_logs", "service": "<service-name>"}
- Search logs: {"action_type": "search_logs", "service": "<service-name>", "query": "<search-term>"}
- Read metrics: {"action_type": "read_metrics", "service": "<service-name>"}
- Read runbook: {"action_type": "read_runbook", "runbook": "<runbook-name>"}
- Diagnose: {"action_type": "diagnose", "root_cause": "<your diagnosis>"}
- Restart service: {"action_type": "restart_service", "service": "<service-name>"}
- Rollback: {"action_type": "rollback", "service": "<service-name>", "version": "previous"}
- Scale up: {"action_type": "scale_up", "service": "<service-name>"}
- Alert on-call: {"action_type": "alert_oncall", "message": "<alert-message>"}
- Acknowledge alert: {"action_type": "acknowledge", "alert_id": "<alert-id>"}
- Block IP range: {"action_type": "block_ip_range", "ip_range": "<cidr>"}
- Create index: {"action_type": "create_index", "table": "<table>", "column": "<column>"}
- Failover: {"action_type": "failover", "service": "<service-name>", "target_region": "us-west-2"}

Strategy:
1. First gather information (read_logs, read_metrics) before acting
2. Diagnose before fixing
3. Fix the ROOT CAUSE, not symptoms
4. Do NOT restart healthy services — this causes penalties

Respond with ONLY a valid JSON object. No explanation, no markdown."""

# Test prompt generation
test_prompt = observation_to_prompt(test_obs, 'easy')
print('Sample prompt (first 800 chars):')
print(test_prompt[:800])
print(f'\nTotal prompt length: {len(test_prompt)} chars')

In [ ]:
# ── Cell 6: Load Model with Unsloth ──────────────────────────────────────────
from unsloth import FastLanguageModel
import torch

print(f'Loading {CONFIG["model_name"]} with Unsloth...')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['model_name'],
    max_seq_length=CONFIG['max_seq_length'],
    dtype=None,  # auto-detect
    load_in_4bit=CONFIG['load_in_4bit'],
    token=hf_token,
)

# Apply LoRA with Unsloth
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG['lora_rank'],
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    use_rslora=False,
)

print(f'\n✅ Model loaded and LoRA applied')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')
print(f'VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
# ── Cell 7: Episode Runner ────────────────────────────────────────────────────
import re

FastLanguageModel.for_inference(model)

def parse_action(text: str) -> dict:
    """Extract JSON action from LLM output. Returns noop on parse failure."""
    text = text.strip()
    # Try to find JSON block
    patterns = [
        r'```json\s*({.*?})\s*```',
        r'```\s*({.*?})\s*```',
        r'({\s*"action_type"[^}]+})',
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1))
            except:
                continue
    # Try raw JSON
    try:
        return json.loads(text)
    except:
        return {'action_type': 'noop'}


def generate_action(obs: dict, task_id: str) -> tuple:
    """Generate an action from the current observation using the LLM."""
    user_content = observation_to_prompt(obs, task_id)

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_content}
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt'
    ).to('cuda')

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=128,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        output[0][input_ids.shape[1]:],
        skip_special_tokens=True
    )

    action = parse_action(generated)
    return action, generated


def run_episode(task_id: str, seed: int = None, verbose: bool = False) -> float:
    """Run one episode and return the final reward score."""
    obs = env_reset(task_id, seed=seed)
    total_reward = 0.0
    done = False

    for step in range(CONFIG['max_steps_per_episode']):
        if done:
            break

        action, raw_output = generate_action(obs, task_id)

        if verbose:
            print(f'  Step {step+1}: {action.get("action_type","?")} '
                  f'(service={action.get("service","-")})')

        result = env_step(action)
        total_reward += result.get('reward', 0.0)
        obs = result.get('observation', obs)
        done = result.get('done', False)

    # Get final graded score
    state = env_state()
    final_score = state.get('current_score', total_reward)
    return final_score


# Test: run one episode before training
print('Testing episode runner (1 episode, verbose)...')
test_score = run_episode('easy', seed=99, verbose=True)
print(f'\n✅ Test episode score: {test_score:.3f}')

In [ ]:
# ── Cell 8: Pre-training Baseline ─────────────────────────────────────────────
import random

print('Running pre-training baseline (10 episodes per task)...')
print('This is the BEFORE score — shows the untrained model.')
print()

baseline_scores = {}
for task_id in CONFIG['tasks']:
    scores = []
    for i in range(10):
        seed = random.randint(0, 9999)
        score = run_episode(task_id, seed=seed)
        scores.append(score)
        print(f'  [{task_id}] Episode {i+1}/10: {score:.3f}', end='\r')

    avg = sum(scores) / len(scores)
    baseline_scores[task_id] = {'scores': scores, 'avg': avg}
    print(f'  [{task_id}] Baseline avg: {avg:.3f} (min={min(scores):.3f}, max={max(scores):.3f})')

print()
print('Baseline summary:')
for task_id, data in baseline_scores.items():
    print(f'  {task_id}: {data["avg"]:.3f}')

In [ ]:
# ── Cell 9: GRPO Setup ────────────────────────────────────────────────────────
# Uses episode-level updates (not per-step) + KL penalty to prevent forgetting.
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
import copy, torch

FastLanguageModel.for_training(model)

# Frozen reference model for KL penalty
ref_model = copy.deepcopy(model)
for p in ref_model.parameters():
    p.requires_grad = False
ref_model.eval()
print('✅ Reference model frozen')

total_episodes = CONFIG['episodes_per_task'] * len(CONFIG['tasks'])
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CONFIG['learning_rate'], weight_decay=0.01
)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, total_episodes // 10),
    num_training_steps=total_episodes
)


def run_episode_collect(task_id, seed):
    """
    KEY FIX: group completions are scored on FRESH env snapshots.
    Only the best action advances the main episode.
    This prevents reward gates from being burned by group generation.
    """
    obs = env_reset(task_id, seed=seed)
    trajectory = []
    done = False

    FastLanguageModel.for_inference(model)

    for step in range(CONFIG['max_steps_per_episode']):
        if done:
            break

        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': observation_to_prompt(obs, task_id)}
        ]
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors='pt'
        ).to('cuda')

        # Step 1: Generate all completions (no env calls yet)
        group_completions, group_texts = [], []
        for _ in range(CONFIG['grpo_group_size']):
            with torch.no_grad():
                out = model.generate(
                    input_ids, max_new_tokens=128, temperature=0.9,
                    do_sample=True, pad_token_id=tokenizer.eos_token_id,
                )
            gen_ids = out[0][input_ids.shape[1]:]
            group_completions.append(gen_ids)
            group_texts.append(tokenizer.decode(gen_ids, skip_special_tokens=True))

        # Step 2: Score each completion on a FRESH env snapshot
        # Each gets its own reset so reward gates are clean per completion
        group_rewards = []
        for gen_text in group_texts:
            action = parse_action(gen_text)
            try:
                env_reset(task_id, seed=seed)  # fresh snapshot
                res = env_step(action)
                r = res.get('reward', 0.0)
            except:
                r = 0.0
            # Exploration bonus: non-noop gets +0.02 to bootstrap learning
            if action.get('action_type', 'noop') != 'noop':
                r += 0.02
            group_rewards.append(r)

        # Step 3: Advance MAIN episode with best action
        best_idx = group_rewards.index(max(group_rewards))
        best_action = parse_action(group_texts[best_idx])
        try:
            # Re-sync: reset to current state then step
            adv_res = env_step(best_action)
            obs = adv_res.get('observation', obs)
            done = adv_res.get('done', False)
        except:
            done = True

        trajectory.append({
            'input_ids': input_ids,
            'completions': group_completions,
            'rewards': group_rewards,
        })

    try:
        state = env_state()
        final_score = state.get('current_score', 0.0)
    except:
        final_score = 0.0

    return trajectory, final_score


def update_from_trajectory(trajectory):
    """Single model update from full episode trajectory with KL penalty."""
    if not trajectory:
        return 0.0

    FastLanguageModel.for_training(model)
    model.train()
    optimizer.zero_grad()

    total_loss = torch.tensor(0.0).to('cuda')

    for step_data in trajectory:
        input_ids = step_data['input_ids']
        completions = step_data['completions']
        rewards = step_data['rewards']

        rewards_t = torch.tensor(rewards, dtype=torch.float32)
        if rewards_t.std() > 1e-8:
            advantages = (rewards_t - rewards_t.mean()) / (rewards_t.std() + 1e-8)
        else:
            advantages = rewards_t - rewards_t.mean()

        best_idx = rewards.index(max(rewards))
        best_ids = completions[best_idx]
        best_adv = advantages[best_idx]

        full_ids = torch.cat([input_ids[0], best_ids]).unsqueeze(0)
        labels = full_ids.clone()
        labels[0, :input_ids.shape[1]] = -100

        outputs = model(full_ids, labels=labels)
        policy_loss = outputs.loss * (-best_adv)

        # KL penalty vs reference model
        with torch.no_grad():
            ref_out = ref_model(full_ids)
        ref_logits = ref_out.logits[:, input_ids.shape[1]-1:-1, :]
        pol_logits = outputs.logits[:, input_ids.shape[1]-1:-1, :]
        kl = torch.nn.functional.kl_div(
            torch.log_softmax(pol_logits, dim=-1),
            torch.softmax(ref_logits, dim=-1),
            reduction='batchmean'
        )
        total_loss = total_loss + policy_loss + CONFIG['kl_coeff'] * kl

    total_loss = total_loss / len(trajectory)
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], 0.5
    )
    optimizer.step()
    scheduler.step()
    return total_loss.item()


print('✅ GRPO setup complete')
print(f'Strategy: fresh env per completion → episode-level update')
print(f'LR={CONFIG["learning_rate"]} | KL={CONFIG["kl_coeff"]} | Groups={CONFIG["grpo_group_size"]}')

In [ ]:
# ── Cell 10: Training Loop ────────────────────────────────────────────────────
import os, time, random, json

os.makedirs(CONFIG['output_dir'], exist_ok=True)
training_log = []
episode_scores = {task: [] for task in CONFIG['tasks']}
global_episode = 0
start_time = time.time()

print('=' * 60)
print('ARIA GRPO TRAINING')
print(f'LR={CONFIG["learning_rate"]} | Groups={CONFIG["grpo_group_size"]} | KL={CONFIG["kl_coeff"]}')
print('=' * 60)

for task_id in CONFIG['tasks']:
    print(f'\n📋 Task: {task_id.upper()} | Baseline: {baseline_scores[task_id]["avg"]:.3f}')
    print('-' * 40)

    for ep in range(CONFIG['episodes_per_task']):
        seed = random.randint(0, 9999)

        trajectory, final_score = run_episode_collect(task_id, seed)
        loss = update_from_trajectory(trajectory)

        episode_scores[task_id].append(final_score)
        global_episode += 1
        elapsed = (time.time() - start_time) / 60
        recent = episode_scores[task_id][-10:]
        rolling = sum(recent) / len(recent)

        training_log.append({
            'episode': global_episode, 'task_id': task_id,
            'score': final_score, 'rolling_avg': rolling,
            'loss': loss, 'elapsed_min': round(elapsed, 1)
        })

        if (ep + 1) % 5 == 0:
            delta = rolling - baseline_scores[task_id]['avg']
            trend = '📈' if delta > 0.02 else '📉' if delta < -0.02 else '➡️'
            print(
                f'  {trend} Ep {ep+1:3d}/{CONFIG["episodes_per_task"]} | '
                f'Score: {final_score:.3f} | Roll-10: {rolling:.3f} | '
                f'vs baseline: {delta:+.3f} | Loss: {loss:.4f} | {elapsed:.0f}m'
            )

        if global_episode % CONFIG['save_every_n_episodes'] == 0:
            ckpt = f'{CONFIG["output_dir"]}/checkpoint-ep{global_episode}'
            model.save_pretrained(ckpt)
            tokenizer.save_pretrained(ckpt)
            print(f'  💾 Checkpoint ep{global_episode}')

    task_avg = sum(episode_scores[task_id]) / len(episode_scores[task_id])
    base_avg = baseline_scores[task_id]['avg']
    delta = task_avg - base_avg
    result = '✅ IMPROVED' if delta > 0.02 else '⚠️ FLAT' if delta > -0.02 else '❌ DEGRADED'
    print(f'\n{result} {task_id}: {base_avg:.3f} → {task_avg:.3f} ({delta:+.3f})')

print(f'\n🎉 Training complete! {(time.time()-start_time)/60:.0f} minutes')

In [ ]:
# ── Cell 11: Post-Training Evaluation ────────────────────────────────────────
FastLanguageModel.for_inference(model)

print('Running post-training evaluation (10 episodes per task)...')
post_scores = {}

for task_id in CONFIG['tasks']:
    scores = []
    for i in range(10):
        seed = random.randint(10000, 19999)  # unseen seeds
        score = run_episode(task_id, seed=seed)
        scores.append(score)

    avg = sum(scores) / len(scores)
    post_scores[task_id] = {'scores': scores, 'avg': avg}
    improvement = avg - baseline_scores[task_id]['avg']
    print(f'  [{task_id}] Post-training avg: {avg:.3f} '
          f'(baseline: {baseline_scores[task_id]["avg"]:.3f}, '
          f'improvement: +{improvement:.3f})')

# Test generalization on unseen tasks
print('\nTesting generalization on UNSEEN tasks...')
unseen_tasks = ['hard', 'bonus']
generalization_scores = {}
for task_id in unseen_tasks:
    scores = []
    for i in range(5):
        seed = random.randint(0, 9999)
        try:
            score = run_episode(task_id, seed=seed)
            scores.append(score)
        except:
            scores.append(0.0)
    avg = sum(scores) / len(scores)
    generalization_scores[task_id] = avg
    print(f'  [{task_id}] Generalization avg: {avg:.3f}')

In [ ]:
# ── Cell 12: Learning Curve Visualization ────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#0d1117')

COLORS = {'easy': '#4caf50', 'medium': '#ff9800', 'hard': '#f44336', 'bonus': '#9c27b0'}

# Plot 1: Training reward curves
ax1 = axes[0]
ax1.set_facecolor('#161b22')
ax1.set_title('GRPO Training Reward Curves', color='white', fontsize=13, fontweight='bold')

for task_id in CONFIG['tasks']:
    task_log = [e for e in training_log if e['task_id'] == task_id]
    episodes = [e['episode'] for e in task_log]
    scores = [e['score'] for e in task_log]

    # Smooth with rolling average
    window = 5
    smoothed = np.convolve(scores, np.ones(window)/window, mode='valid')
    ep_smooth = episodes[window-1:]

    color = COLORS.get(task_id, '#58a6ff')
    ax1.plot(episodes, scores, alpha=0.2, color=color, linewidth=1)
    ax1.plot(ep_smooth, smoothed, color=color, linewidth=2.5,
             label=f'{task_id} (smoothed)')

ax1.set_xlabel('Episode', color='#8b949e')
ax1.set_ylabel('Reward Score', color='#8b949e')
ax1.tick_params(colors='#8b949e')
ax1.spines['bottom'].set_color('#30363d')
ax1.spines['left'].set_color('#30363d')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.legend(facecolor='#161b22', labelcolor='white', fontsize=10)
ax1.set_ylim(0, 1.05)
ax1.grid(True, alpha=0.1, color='#30363d')

# Plot 2: Before vs After bar chart
ax2 = axes[1]
ax2.set_facecolor('#161b22')
ax2.set_title('Before vs After Training', color='white', fontsize=13, fontweight='bold')

all_tasks = CONFIG['tasks']
x = np.arange(len(all_tasks))
width = 0.35

before_vals = [baseline_scores[t]['avg'] for t in all_tasks]
after_vals = [post_scores[t]['avg'] for t in all_tasks]

bars1 = ax2.bar(x - width/2, before_vals, width, label='Before Training',
                color='#f85149', alpha=0.8, edgecolor='none')
bars2 = ax2.bar(x + width/2, after_vals, width, label='After Training',
                color='#3fb950', alpha=0.8, edgecolor='none')

for bar, val in zip(bars1, before_vals):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{val:.2f}', ha='center', va='bottom', color='white', fontsize=9)
for bar, val in zip(bars2, after_vals):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{val:.2f}', ha='center', va='bottom', color='white', fontsize=9)

ax2.set_xticks(x)
ax2.set_xticklabels(all_tasks, color='#8b949e')
ax2.tick_params(colors='#8b949e')
ax2.spines['bottom'].set_color('#30363d')
ax2.spines['left'].set_color('#30363d')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.legend(facecolor='#161b22', labelcolor='white', fontsize=10)
ax2.set_ylim(0, 1.1)
ax2.set_ylabel('Average Score', color='#8b949e')
ax2.grid(True, alpha=0.1, color='#30363d', axis='y')

# Plot 3: Summary stats
ax3 = axes[2]
ax3.set_facecolor('#161b22')
ax3.set_title('Training Summary', color='white', fontsize=13, fontweight='bold')
ax3.axis('off')

summary_lines = [
    ('Model', CONFIG['model_name'].split('/')[-1]),
    ('Algorithm', 'GRPO (Group Relative PO)'),
    ('Fine-tuning', 'Unsloth LoRA 4-bit'),
    ('Total Episodes', str(global_episode)),
    ('', ''),
]
for task_id in CONFIG['tasks']:
    before = baseline_scores[task_id]['avg']
    after = post_scores[task_id]['avg']
    summary_lines.append((f'{task_id} improvement',
                           f'{before:.2f} → {after:.2f} (+{after-before:.2f})'))

if generalization_scores:
    summary_lines.append(('', ''))
    summary_lines.append(('Generalization', ''))
    for task_id, score in generalization_scores.items():
        summary_lines.append((f'  {task_id} (unseen)', f'{score:.2f}'))

y_pos = 0.95
for label, value in summary_lines:
    if label == '':
        y_pos -= 0.05
        continue
    ax3.text(0.05, y_pos, label + ':', color='#8b949e', fontsize=10,
             transform=ax3.transAxes, fontweight='bold')
    ax3.text(0.55, y_pos, value, color='#c9d1d9', fontsize=10,
             transform=ax3.transAxes)
    y_pos -= 0.08

plt.tight_layout()
plt.savefig('training_curve.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
print('✅ Saved training_curve.png')
plt.show()

In [ ]:
# ── Cell 13: Save Weights to HuggingFace Hub ─────────────────────────────────
from huggingface_hub import HfApi

print(f'Saving model to HuggingFace Hub: {CONFIG["hf_repo"]}')
print('This may take 5-10 minutes...')

# Save merged model (LoRA merged into base)
model.save_pretrained_merged(
    CONFIG['output_dir'],
    tokenizer,
    save_method='merged_16bit',
)

# Push to Hub
model.push_to_hub_merged(
    CONFIG['hf_repo'],
    tokenizer,
    save_method='merged_16bit',
    token=hf_token,
)

print(f'\n✅ Model pushed to: https://huggingface.co/{CONFIG["hf_repo"]}')

# Also push training curve
api = HfApi()
api.upload_file(
    path_or_fileobj='training_curve.png',
    path_in_repo='training_curve.png',
    repo_id=CONFIG['hf_repo'],
    token=hf_token,
)
print('✅ training_curve.png uploaded to Hub')

# Save training log as JSON
import json
with open('training_log.json', 'w') as f:
    json.dump(training_log, f, indent=2)
api.upload_file(
    path_or_fileobj='training_log.json',
    path_in_repo='training_log.json',
    repo_id=CONFIG['hf_repo'],
    token=hf_token,
)
print('✅ training_log.json uploaded')
print(f'\n🎉 Everything saved. Your fine-tuned model is live at:')
print(f'   https://huggingface.co/{CONFIG["hf_repo"]}')

## Results Summary

| Metric | Value |
|--------|-------|
| Model | Llama-3.2-3B-Instruct (Unsloth 4-bit LoRA) |
| Algorithm | GRPO — episode-level updates + KL penalty |
| Tasks trained | easy, medium |
| Total episodes | 160 |
| Key fix | Fresh env per group completion — reward gates not burned |
| Weights | `https://huggingface.co/Arijit-07/aria-devops-llama3b` |

In [1]:
import os
os.environ['UNSLOTH_RETURN_LOGITS'] = '1'

CHECKPOINT = r'D:\My Projects\devops-incident-env\kaggle_training\results\aria-llama3b\checkpoint-ep140'
HF_TOKEN = 'YOUR_HF_WRITE_TOKEN_HERE'  # ← paste your write token
HF_REPO = 'Arijit-07/aria-devops-llama3b'
BASE_URL = 'https://arijit-07-devops-incident-response.hf.space'

print('✅ Config set')
print(f'Checkpoint: {CHECKPOINT}')
print(f'Exists: {os.path.exists(CHECKPOINT)}')

✅ Config set
Checkpoint: D:\My Projects\devops-incident-env\kaggle_training\results\aria-llama3b\checkpoint-ep140
Exists: True


In [2]:
import subprocess
subprocess.run(['pip', 'install', '-q',
    'unsloth',
    'transformers>=4.48.0',
    'peft', 'accelerate', 'bitsandbytes',
    'requests', 'matplotlib', 'huggingface_hub', 'torch'
], capture_output=True)

import unsloth, transformers, torch
print(f'✅ unsloth {unsloth.__version__}')
print(f'✅ transformers {transformers.__version__}')
print(f'✅ torch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [ ]:
from unsloth import FastLanguageModel

print('Loading fine-tuned checkpoint...')
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=CHECKPOINT,
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(ft_model)
print('✅ Fine-tuned model loaded')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
print('Loading base model for comparison...')
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-3B-Instruct',
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)
print('✅ Base model loaded')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
print('Loading base model for comparison...')
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-3B-Instruct',
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)
print('✅ Base model loaded')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
import requests, json, re, random

def env_reset(task_id, seed=None):
    payload = {'task_id': task_id}
    if seed is not None: payload['seed'] = seed
    for attempt in range(3):
        try:
            r = requests.post(f'{BASE_URL}/reset', json=payload, timeout=30)
            r.raise_for_status()
            return r.json()
        except:
            if attempt == 2: raise
            import time; time.sleep(5)

def env_step(action):
    for attempt in range(3):
        try:
            r = requests.post(f'{BASE_URL}/step', json=action, timeout=30)
            r.raise_for_status()
            return r.json()
        except:
            if attempt == 2: raise
            import time; time.sleep(5)

def env_state():
    r = requests.get(f'{BASE_URL}/state', timeout=30)
    r.raise_for_status()
    return r.json()

SYSTEM_PROMPT = """You are an expert DevOps engineer responding to a production incident.
Respond with ONLY a valid JSON action object. No explanation, no markdown.
Available actions:
- {"action_type": "read_logs", "service": "<name>"}
- {"action_type": "read_metrics", "service": "<name>"}
- {"action_type": "search_logs", "service": "<name>", "query": "<term>"}
- {"action_type": "diagnose", "root_cause": "<diagnosis>"}
- {"action_type": "restart_service", "service": "<name>"}
- {"action_type": "rollback", "service": "<name>", "version": "previous"}
- {"action_type": "alert_oncall", "message": "<msg>"}
- {"action_type": "noop"}"""

def obs_to_text(obs, task_id):
    lines = [f'=== INCIDENT | Task: {task_id.upper()} | Step: {obs.get("step",0)}/{obs.get("max_steps",15)} ===', '']
    for a in sorted(obs.get('active_alerts', []),
                    key=lambda x: x.get('severity',''), reverse=True):
        lines.append(f'ALERT [{a.get("severity","").upper()}] {a.get("service","")}: {a.get("message","")}')
    lines.append('')
    for s in sorted(obs.get('services', []),
                    key=lambda x: x.get('error_rate',0), reverse=True):
        lines.append(
            f'SERVICE {s.get("name",""):28s} | {s.get("status",""):10s} | '
            f'err={s.get("error_rate",0):.3f} | mem={s.get("memory",0):.1f}%'
        )
    evidence = obs.get('evidence_log', [])
    if evidence:
        lines.append('')
        lines.append('EVIDENCE:')
        for e in evidence[-3:]:
            lines.append(f'  [{e.get("action_type","").upper()}] {e.get("content","")[:150]}')
    return '\n'.join(lines)

def parse_action(text):
    text = text.strip()
    for pat in [
        r'```json\s*({.*?})\s*```',
        r'```\s*({.*?})\s*```',
        r'({\s*"action_type"[^}]+})',
    ]:
        m = re.search(pat, text, re.DOTALL)
        if m:
            try: return json.loads(m.group(1))
            except: continue
    try: return json.loads(text)
    except: return {'action_type': 'noop'}

def run_episode(m, tok, task_id, seed, verbose=False):
    obs = env_reset(task_id, seed=seed)
    done = False
    for step in range(15):
        if done: break
        msgs = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': obs_to_text(obs, task_id)}
        ]
        ids = tok.apply_chat_template(
            msgs, tokenize=True, add_generation_prompt=True,
            return_tensors='pt'
        ).to('cuda')
        with torch.no_grad():
            out = m.generate(
                ids, max_new_tokens=100, temperature=0.3,
                do_sample=True, pad_token_id=tok.eos_token_id,
            )
        text = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
        action = parse_action(text)
        if verbose:
            print(f'  Step {step+1}: {action}')
        result = env_step(action)
        obs = result.get('observation', obs)
        done = result.get('done', False)
    return env_state().get('current_score', 0.0)

# Test connection
health = requests.get(f'{BASE_URL}/health', timeout=15).json()
print(f'✅ Environment: {health}')
print('✅ All helpers ready')

In [ ]:
SEEDS = [50001, 50008, 50015, 50022, 50029,
         50036, 50043, 50050, 50057, 50064]

results = {}
print('Running evaluation — 10 episodes per model per task')
print('='*60)

for task_id in ['easy', 'medium']:
    print(f'\nTask: {task_id.upper()}')
    print('-'*40)
    base_scores, ft_scores = [], []

    for seed in SEEDS:
        bs = run_episode(base_model, base_tokenizer, task_id, seed)
        fs = run_episode(ft_model, ft_tokenizer, task_id, seed)
        base_scores.append(bs)
        ft_scores.append(fs)
        print(f'  seed={seed} | base={bs:.3f} | fine-tuned={fs:.3f} | Δ={fs-bs:+.3f}')

    base_avg = sum(base_scores)/len(base_scores)
    ft_avg = sum(ft_scores)/len(ft_scores)
    delta = ft_avg - base_avg
    results[task_id] = {
        'base_scores': base_scores,
        'ft_scores': ft_scores,
        'base_avg': base_avg,
        'ft_avg': ft_avg,
        'delta': delta
    }
    symbol = '✅ IMPROVED' if delta > 0.02 else '⚠️ FLAT' if delta > -0.02 else '❌ DEGRADED'
    print(f'\n{symbol}')
    print(f'  Base avg:       {base_avg:.3f}')
    print(f'  Fine-tuned avg: {ft_avg:.3f}')
    print(f'  Improvement:    {delta:+.3f}')

print('\n' + '='*60)
print('FINAL RESULTS')
print('='*60)
for task_id, r in results.items():
    print(f'{task_id}: {r["base_avg"]:.3f} → {r["ft_avg"]:.3f} ({r["delta"]:+.3f})')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0d1117')

COLORS = {'base': '#f85149', 'ft': '#3fb950'}

for idx, task_id in enumerate(['easy', 'medium']):
    ax = axes[idx]
    ax.set_facecolor('#161b22')
    r = results[task_id]

    x = np.arange(len(SEEDS))
    w = 0.35
    b = ax.bar(x - w/2, r['base_scores'], w,
               label='Base model', color='#f85149', alpha=0.85)
    f = ax.bar(x + w/2, r['ft_scores'], w,
               label='Fine-tuned (ep140)', color='#3fb950', alpha=0.85)

    # Value labels
    for bar in b:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2., h+0.01,
                f'{h:.2f}', ha='center', color='white', fontsize=7)
    for bar in f:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2., h+0.01,
                f'{h:.2f}', ha='center', color='white', fontsize=7)

    # Avg lines
    ax.axhline(y=r['base_avg'], color='#f85149', linestyle='--',
               linewidth=1.5, alpha=0.6, label=f'Base avg: {r["base_avg"]:.3f}')
    ax.axhline(y=r['ft_avg'], color='#3fb950', linestyle='--',
               linewidth=1.5, alpha=0.6, label=f'FT avg: {r["ft_avg"]:.3f}')

    ax.set_title(
        f'Task: {task_id.upper()} | Improvement: {r["delta"]:+.3f}',
        color='white', fontsize=13, fontweight='bold'
    )
    ax.set_xticks(x)
    ax.set_xticklabels([f's{i+1}' for i in range(len(SEEDS))],
                       color='#8b949e', fontsize=8)
    ax.tick_params(colors='#8b949e')
    for spine in ax.spines.values(): spine.set_color('#30363d')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score', color='#8b949e')
    ax.set_xlabel('Episode (unseen seeds)', color='#8b949e')
    ax.legend(facecolor='#161b22', labelcolor='white', fontsize=9)
    ax.grid(True, alpha=0.1, color='#30363d', axis='y')

fig.suptitle(
    'ARIA — GRPO Fine-tuning Results\nLlama-3.2-3B | 140 episodes | easy + medium tasks',
    color='white', fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('training_curve.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
print('✅ Saved training_curve.png')
plt.show()

In [ ]:
from huggingface_hub import HfApi, login
import os

login(token=HF_TOKEN)
api = HfApi()

# Create repo
api.create_repo(
    repo_id=HF_REPO,
    repo_type='model',
    exist_ok=True,
    token=HF_TOKEN,
)
print(f'✅ Repo ready: {HF_REPO}')

# Upload checkpoint files
print('Uploading adapter weights...')
for filename in os.listdir(CHECKPOINT):
    filepath = os.path.join(CHECKPOINT, filename)
    if os.path.isfile(filepath):
        api.upload_file(
            path_or_fileobj=filepath,
            path_in_repo=filename,
            repo_id=HF_REPO,
            token=HF_TOKEN,
        )
        print(f'  ✅ {filename}')

# Upload training curve
api.upload_file(
    path_or_fileobj='training_curve.png',
    path_in_repo='training_curve.png',
    repo_id=HF_REPO,
    token=HF_TOKEN,
)
print('  ✅ training_curve.png')

print(f'\n🎉 Everything live at: https://huggingface.co/{HF_REPO}')

In [ ]:
easy_r = results['easy']
medium_r = results['medium']

model_card = f"""---
base_model: unsloth/Llama-3.2-3B-Instruct
library_name: peft
pipeline_tag: text-generation
tags:
- lora
- unsloth
- grpo
- reinforcement-learning
- devops
- incident-response
---

# ARIA — DevOps Incident Response Agent
## Llama-3.2-3B fine-tuned with GRPO

Fine-tuned on the [ARIA DevOps Incident Response](https://huggingface.co/spaces/Arijit-07/devops-incident-response) 
RL environment using Group Relative Policy Optimization (GRPO).

## Training

- **Algorithm:** GRPO (Group Relative Policy Optimization)
- **Base model:** Llama-3.2-3B-Instruct
- **Fine-tuning:** Unsloth LoRA (rank=16, alpha=32, 4-bit quantized)
- **Episodes:** 140 (easy + medium tasks)
- **Environment:** Live DevOps incident response simulation

## Results (10 unseen episodes per task)

| Task | Base Model | Fine-tuned | Improvement |
|------|-----------|------------|-------------|
| easy | {easy_r['base_avg']:.3f} | {easy_r['ft_avg']:.3f} | {easy_r['delta']:+.3f} |
| medium | {medium_r['base_avg']:.3f} | {medium_r['ft_avg']:.3f} | {medium_r['delta']:+.3f} |

## Environment

The agent learns to diagnose and fix production incidents:
- 7 task types: OOM crashes, cascading failures, silent corruption, DDoS, DB degradation, multi-region failover
- 14 action types including read_logs, diagnose, restart_service, rollback
- Dense reward shaping with collateral damage penalties
- Partial log observability — agents must learn to query

## Usage

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained("unsloth/Llama-3.2-3B-Instruct")
model = PeftModel.from_pretrained(base, "Arijit-07/aria-devops-llama3b")
tokenizer = AutoTokenizer.from_pretrained("Arijit-07/aria-devops-llama3b")
```

## Links
- Environment: https://huggingface.co/spaces/Arijit-07/devops-incident-response
- API docs: https://arijit-07-devops-incident-response.hf.space/docs
"""

with open('README.md', 'w') as f:
    f.write(model_card)

api.upload_file(
    path_or_fileobj='README.md',
    path_in_repo='README.md',
    repo_id=HF_REPO,
    token=HF_TOKEN,
)
print('✅ Model card uploaded')
print(f'🎉 Complete: https://huggingface.co/{HF_REPO}')

In [5]:
HF_TOKEN = 'YOUR_HF_WRITE_TOKEN_HERE'

In [8]:
from huggingface_hub import HfApi, login

HF_TOKEN = 'YOUR_HF_WRITE_TOKEN_HERE'  # ← paste your REAL token here
                                            # it must start with hf_

login(token=HF_TOKEN, add_to_git_credential=False)
print('✅ Logged in')

✅ Logged in


In [9]:
from huggingface_hub import HfApi
import os

HF_REPO = 'Arijit-07/aria-devops-llama3b'
CHECKPOINT = r'D:\My Projects\devops-incident-env\kaggle_training\results\aria-llama3b\checkpoint-ep140'

api = HfApi()

api.create_repo(repo_id=HF_REPO, repo_type='model', exist_ok=True)
print(f'✅ Repo ready')

for filename in os.listdir(CHECKPOINT):
    filepath = os.path.join(CHECKPOINT, filename)
    if os.path.isfile(filepath):
        api.upload_file(
            path_or_fileobj=filepath,
            path_in_repo=filename,
            repo_id=HF_REPO,
            token=HF_TOKEN,
        )
        print(f'✅ {filename}')

print(f'\n🎉 Live at: https://huggingface.co/{HF_REPO}')

✅ Repo ready
✅ adapter_config.json


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ adapter_model.safetensors


No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


✅ chat_template.jinja
✅ README.md


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


✅ tokenizer.json
✅ tokenizer_config.json

🎉 Live at: https://huggingface.co/Arijit-07/aria-devops-llama3b


In [10]:
model_card = f"""---
base_model: unsloth/Llama-3.2-3B-Instruct
library_name: peft
pipeline_tag: text-generation
tags:
- lora
- unsloth
- grpo
- reinforcement-learning
- devops
- incident-response
---

# ARIA — DevOps Incident Response Agent
### Llama-3.2-3B fine-tuned with GRPO

Fine-tuned on the [ARIA DevOps Incident Response](https://huggingface.co/spaces/Arijit-07/devops-incident-response) 
RL environment using Group Relative Policy Optimization (GRPO).

## Training Details

- **Algorithm:** GRPO (Group Relative Policy Optimization)
- **Base model:** Llama-3.2-3B-Instruct
- **Fine-tuning:** Unsloth LoRA (rank=16, alpha=32, 4-bit quantized)
- **Episodes:** 140 across easy + medium tasks
- **Training time:** ~10 hours on Kaggle T4 x2
- **Environment:** Live DevOps incident response simulation

## What the Agent Learns

The agent is trained to respond to production software incidents by:
1. Gathering information (read_logs, read_metrics, search_logs)
2. Diagnosing the root cause before acting
3. Applying the correct fix (restart, rollback, scale_up, block_ip etc.)
4. Avoiding collateral damage to healthy services

## Environment

7 task types of escalating difficulty:
- **Easy:** Single service OOM crash-loop
- **Medium:** Cascading connection pool failure
- **Hard:** Silent data corruption (all services green)
- **Bonus:** Two simultaneous independent failures
- **Security:** DDoS botnet credential stuffing
- **Database:** Missing index causing full table scans
- **Failover:** Multi-region network partition

14 action types · Dense reward shaping · Partial log observability · SLA degradation per step

## Links
- **Live environment:** https://huggingface.co/spaces/Arijit-07/devops-incident-response
- **Interactive API:** https://arijit-07-devops-incident-response.hf.space/docs
- **GitHub:** https://github.com/Twilight-13/devops-incident-response

## Usage

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained(
    "unsloth/Llama-3.2-3B-Instruct",
    load_in_4bit=True
)
model = PeftModel.from_pretrained(base, "Arijit-07/aria-devops-llama3b")
tokenizer = AutoTokenizer.from_pretrained("Arijit-07/aria-devops-llama3b")
```
"""

from huggingface_hub import HfApi
api = HfApi()

with open('README.md', 'w', encoding='utf-8') as f:
    f.write(model_card)

api.upload_file(
    path_or_fileobj='README.md',
    path_in_repo='README.md',
    repo_id='Arijit-07/aria-devops-llama3b',
    token=HF_TOKEN,
)
print('✅ Model card uploaded')
print('🎉 https://huggingface.co/Arijit-07/aria-devops-llama3b')

✅ Model card uploaded
🎉 https://huggingface.co/Arijit-07/aria-devops-llama3b
